In [8]:
import numpy as np
import pandas as pd

## Phase 2 : Data Cleaning and Portfolio Construction ( Buy and Hold )

In [9]:
# Step 0: Loading Phase 1 outputs
prices = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\Raw_Portfolio_Price_stratified.csv', index_col = 0, parse_dates = True)
benchmark_df = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\Benchmark_prices_1.csv', index_col = 0, parse_dates = True)
# Single column: Plain Series
benchmark_prices = benchmark_df.iloc[:,0]
print(f'Loaded {prices.shape[1]} tickers and {len(prices)} raw rows')

Loaded 50 tickers and 1507 raw rows


In [10]:
# Step 1: Establishing a common usable window 
first_valid_dates = prices.apply(lambda col: col.first_valid_index())
common_start = first_valid_dates.max()
print(f'Latest first valid date among all tickers {common_start.date()}')

laggards = first_valid_dates[first_valid_dates == common_start]
print('Tickers that set this cutoff')
print(laggards)

trimmed_prices = prices.loc[common_start:]
trimmed_benchmark = benchmark_prices.loc[common_start:]


Latest first valid date among all tickers 2020-07-24
Tickers that set this cutoff
AFL     2020-07-24
AON     2020-07-24
APA     2020-07-24
APD     2020-07-24
BA      2020-07-24
BKNG    2020-07-24
BKR     2020-07-24
CI      2020-07-24
CINF    2020-07-24
CMI     2020-07-24
COF     2020-07-24
D       2020-07-24
DVA     2020-07-24
EA      2020-07-24
ECL     2020-07-24
ED      2020-07-24
EQR     2020-07-24
ESS     2020-07-24
ETN     2020-07-24
GM      2020-07-24
GOOGL   2020-07-24
HST     2020-07-24
INTU    2020-07-24
KLAC    2020-07-24
KMB     2020-07-24
KR      2020-07-24
L       2020-07-24
LLY     2020-07-24
LRCX    2020-07-24
MAS     2020-07-24
MLM     2020-07-24
MMM     2020-07-24
NDAQ    2020-07-24
NOW     2020-07-24
PM      2020-07-24
PWR     2020-07-24
RF      2020-07-24
ROP     2020-07-24
ROST    2020-07-24
RVTY    2020-07-24
SO      2020-07-24
TFC     2020-07-24
TGT     2020-07-24
TPR     2020-07-24
TXN     2020-07-24
TXT     2020-07-24
ULTA    2020-07-24
URI     2020-07-24
VRTX  

In [12]:
# Step 2: Defensive Imputation (Fill only small gaps)
print('--- Missing Value Check and Defensive Imputation ---')
# Checking the current status of the data
initial_missing = trimmed_prices.isna().sum().sum() 
if initial_missing == 0:
    print('There is no missing value in the current historical window')
    print('Executing bounded forward fill ( limit = 3) regardless as a defensive measure for future live data')
else:
    print('Found {len(initial_missing)} missing data points. Executing bounded forward fill')

# Applying forward fill
ffill_limit = 3
filled_prices = trimmed_prices.ffill(limit = ffill_limit)

remaining_missing = filled_prices.isna().sum()
tickers_with_gaps = remaining_missing[remaining_missing > 0]

if len(tickers_with_gaps) > 0:
    print(f'{len(tickers_with_gaps)} ticker(s) still have gaps longer than '
          f'{ffill_limit} days after filling. Excluding these tickers rather than '
          f'dropping shared rows for everyone else:')
    print(tickers_with_gaps)
    clean_prices = filled_prices.drop(columns = tickers_with_gaps.index.tolist())
else:
    print(f'\nNo tickers had gaps longer than {ffill_limit} days. None excluded.')
    clean_prices = filled_prices

clean_benchmark = trimmed_benchmark.ffill(limit = ffill_limit).dropna()

print(f'Final clean universe: {clean_prices.shape[1]} tickers, {len(clean_prices)} trading days')

--- Missing Value Check and Defensive Imputation ---
There is no missing value in the current historical window
Executing bounded forward fill ( limit = 3) regardless as a defensive measure for future live data

No tickers had gaps longer than 3 days. None excluded.
Final clean universe: 50 tickers, 1507 trading days


In [15]:
#Step 3: Flagging stale price and implausible moves
def max_consecutive_repeats(series):
    is_repeat = series == series.shift(1)
    run_id = (~is_repeat).cumsum()
    return is_repeat.groupby(run_id).sum().max()

Stale_Run_Threshold = 5
stale_run = clean_prices.apply(max_consecutive_repeats)
flagged_stale = stale_run[stale_run >= Stale_Run_Threshold]
if len(flagged_stale) > 0:
    print(f"\nMONITOR (not excluded) - ticker(s) with {Stale_Run_Threshold}+ identical "
          f"consecutive prices (possible stale/no-trade days):")
    print(flagged_stale)

check_returns = clean_prices.pct_change()
Extreme_move_threshold = 0.4 # 40% single day change. Flaging for error check. Don't autoclip

for tickers in clean_prices.columns:
    extreme_days = check_returns[tickers][check_returns[tickers].abs() > Extreme_move_threshold]
    if not extreme_days.empty:
        print(f'n\Flaging for manual review : {ticker} moves over'
              f'{Extreme_move_threshold * 100:.0f}%')
        print(extreme_days)


In [16]:
# Step 4: Calculating return of each stock and the benchmark
return_matrix = np.log(clean_prices/clean_prices.shift(1)).dropna()
benchmark_returns = np.log(clean_benchmark/clean_benchmark.shift(1)).dropna()

In [17]:
# Step 5: Building the portfolio. BUY and HOLD, no daily rebalancing
# Equal weighted portfolio and weights are assigned on day 0 after that left to drift natuarally along stock's own performance.
num_assets = clean_prices.shape[1]
initial_weights = pd.Series(np.repeat(1/num_assets, num_assets), index = clean_prices.columns)

# Price relative: Growth of $1 invested in each stock since day 0
price_relative = clean_prices/clean_prices.iloc[0]

# Portfolio value = weighted sum of each stock's own growth path.
portfolio_value = (price_relative * initial_weights).sum(axis=1)
portfolio_value.name = 'Portfolio_Value'

# After aggregation, taking the log return
portfolio_return = np.log(portfolio_value / portfolio_value.shift(1)).dropna()
portfolio_return.name = 'Portfolio Return'

In [18]:
# Initial Capital and daily P&L
Initial_Capital = 1000000
portfolio_dollar_value = portfolio_value * Initial_Capital
portfolio_dollar_value.name = 'Portfolio_Dollar_Value'

# Daily P&L : Day over day actual change in the dollar value
portfolio_pnl = portfolio_dollar_value.diff().fillna(0)
portfolio_pnl.name = 'Portfolio_PnL'

In [20]:
# Step 6: Rolling Volatility of the portfolio
rolling_volatility = portfolio_return.rolling(window =21).std()*np.sqrt(252)
rolling_volatility.name = 'Rolling_Volatility_Annualized'

In [21]:
# Step 7: Diagnostic - how far have weights drifted from equal-weight?
final_effective_weights = (price_relative.iloc[-1] * initial_weights) / portfolio_value.iloc[-1]
print("\nEffective weights at end of window (buy-and-hold drift from equal-weight):")
print(final_effective_weights.sort_values(ascending=False))
 


Effective weights at end of window (buy-and-hold drift from equal-weight):
PWR      0.094286
TPR      0.073050
KLAC     0.072325
LRCX     0.056986
LLY      0.048528
URI      0.044602
ETN      0.029044
GOOGL    0.025928
BKR      0.024807
CMI      0.024617
AFL      0.023834
RF       0.021825
COF      0.020900
PM       0.020326
GM       0.019868
L        0.019835
APA      0.018078
HST      0.017956
ROST     0.017843
TXT      0.016972
DVA      0.016341
BKNG     0.016142
MLM      0.015766
CINF     0.015472
TXN      0.015459
ULTA     0.014144
NDAQ     0.013927
SO       0.013486
KR       0.011284
ED       0.011238
AON      0.011065
TFC      0.010948
CI       0.010719
VRTX     0.010390
ESS      0.010292
MAS      0.009834
MMM      0.009674
EA       0.009655
EQR      0.009569
ECL      0.008272
TGT      0.007923
BA       0.007309
APD      0.007166
NOW      0.007041
D        0.006952
INTU     0.006492
RVTY     0.006152
KMB      0.005643
ROP      0.005564
ZBH      0.004469
dtype: float64


In [23]:
# Step 8: Save outputs for Phase 3 onward
clean_prices.to_csv('clean_prices.csv')
return_matrix.to_csv('return_matrix.csv')
portfolio_value.to_csv('portfolio_value.csv')
portfolio_dollar_value.to_csv('portfolio_dollar_value.csv')
portfolio_pnl.to_csv('portfolio_pnl.csv')
portfolio_return.to_csv('portfolio_returns.csv')
rolling_volatility.to_csv('rolling_volatility.csv')
 
print("\nSaved: clean_prices.csv, return_matrix.csv, portfolio_value.csv, "
      "portfolio_dollar_value.csv, portfolio_pnl.csv, portfolio_returns.csv, "
      "rolling_volatility.csv")


Saved: clean_prices.csv, return_matrix.csv, portfolio_value.csv, portfolio_dollar_value.csv, portfolio_pnl.csv, portfolio_returns.csv, rolling_volatility.csv
